In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

In [22]:
import numpy as np
import polars as pl
import pandas as pd

# 📌 Charger et transformer le dataset
train = (pl.scan_parquet("./Data/Zzzs_train.parquet")
          .with_columns(
              (pl.col("timestamp")
                 .str.strptime(pl.Datetime, "%Y-%m-%dT%H:%M:%S%Z")
                 .dt.hour()
                 .alias("hour")  # Extraire l'heure
              )
          )
          .drop("timestamp")  # Supprimer timestamp original
          .collect()
          .to_pandas()
)

# 📌 Ajouter encodage sinusoïdal du temps
train['hour_sin'] = np.sin(2 * np.pi * train['hour'] / 24)
train['hour_cos'] = np.cos(2 * np.pi * train['hour'] / 24)
train = train.drop(columns=['hour'])  # Supprimer la colonne brute
df = train.copy()
# 📌 Affichage des premières lignes
print(train.head())


      series_id  step     anglez    enmo  awake  hour_sin  hour_cos
0  08db4255286f     0 -30.845301  0.0447      1       0.5 -0.866025
1  08db4255286f     1 -34.181801  0.0443      1       0.5 -0.866025
2  08db4255286f     2 -33.877102  0.0483      1       0.5 -0.866025
3  08db4255286f     3 -34.282101  0.0680      1       0.5 -0.866025
4  08db4255286f     4 -34.385799  0.0768      1       0.5 -0.866025


In [16]:
import numpy as np
import pandas as pd
from copy import deepcopy
import random
from tqdm import tqdm

def augment_series(df, n_augmentations=2, noise_level=0.05, 
                   time_shift_max=5, drop_prob=0.01):
    """
    Applique des techniques de data augmentation aux séries temporelles.
    
    Args:
        df: DataFrame d'origine contenant les données
        n_augmentations: Nombre de versions augmentées à créer par série
        noise_level: Niveau de bruit gaussien à ajouter (écart-type)
        time_shift_max: Décalage temporel maximal à appliquer
        drop_prob: Probabilité de supprimer un point (pour simuler des données manquantes)
    
    Returns:
        DataFrame augmenté avec de nouvelles séries
    """
    # Créer un DataFrame pour stocker les données augmentées
    augmented_df = pd.DataFrame()
    
    # Traiter chaque série individuellement
    series_ids = df['series_id'].unique()
    print(f"Augmentation de {len(series_ids)} séries...")
    
    for series_id in tqdm(series_ids):
        # Extraire la série originale
        original_series = df[df['series_id'] == series_id].copy()
        
        # Créer n_augmentations nouvelles versions
        for i in range(n_augmentations):
            # Créer une copie de la série originale
            new_series = original_series.copy()
            
            # Générer un nouveau series_id
            new_series_id = f"{series_id}_aug_{i+1}"
            new_series['series_id'] = new_series_id
            
            # Appliquer le bruit gaussien à anglez et enmo
            anglez_noise = np.random.normal(0, noise_level * new_series['anglez'].std(), size=len(new_series))
            enmo_noise = np.random.normal(0, noise_level * new_series['enmo'].std(), size=len(new_series))
            
            new_series['anglez'] = new_series['anglez'] + anglez_noise
            new_series['enmo'] = new_series['enmo'] + enmo_noise
            
            # Appliquer un décalage temporel (modifie légèrement la phase des signaux)
            time_shift = random.randint(-time_shift_max, time_shift_max)
            if time_shift != 0:
                # Pour anglez
                new_series['anglez'] = np.roll(new_series['anglez'].values, time_shift)
                # Pour enmo
                new_series['enmo'] = np.roll(new_series['enmo'].values, time_shift)
            
            # Simuler des données manquantes (remplacer par la moyenne)
            mask = np.random.random(size=len(new_series)) < drop_prob
            if mask.any():
                anglez_mean = new_series['anglez'].mean()
                enmo_mean = new_series['enmo'].mean()
                
                new_series.loc[mask, 'anglez'] = anglez_mean
                new_series.loc[mask, 'enmo'] = enmo_mean
            
            # Ajouter la série augmentée au DataFrame final
            augmented_df = pd.concat([augmented_df, new_series], ignore_index=True)
    
    # Combiner le DataFrame original avec les données augmentées
    combined_df = pd.concat([df, augmented_df], ignore_index=True)
    
    print(f"Taille du DataFrame original: {len(df)}")
    print(f"Taille du DataFrame augmenté: {len(combined_df)}")
    print(f"Nouvelles séries créées: {len(augmented_df) // len(original_series)}")
    
    return combined_df

# Utilisation de la fonction
# augmented_train = augment_series(train, n_augmentations=2, noise_level=0.05)

In [17]:
# appliquer
train = augment_series(train, n_augmentations=2, noise_level=0.05)
print(train.head())



Augmentation de 35 séries...


100%|██████████| 35/35 [00:58<00:00,  1.67s/it]


Taille du DataFrame original: 13165560
Taille du DataFrame augmenté: 39496680
Nouvelles séries créées: 66
      series_id  step     anglez    enmo  awake  hour_sin  hour_cos
0  08db4255286f     0 -30.845301  0.0447      1       0.5 -0.866025
1  08db4255286f     1 -34.181801  0.0443      1       0.5 -0.866025
2  08db4255286f     2 -33.877102  0.0483      1       0.5 -0.866025
3  08db4255286f     3 -34.282101  0.0680      1       0.5 -0.866025
4  08db4255286f     4 -34.385799  0.0768      1       0.5 -0.866025


In [18]:
# Enregistrer le DataFrame augmenté
train.to_parquet("./Zzzzs_augmented.parquet",index=False)
